In [1]:
!pip install -qU langgraph langchain-openai langchain-google-genai langchain-community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.0/256.0 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.1/146.1 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.7/81.7 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.3/573.3 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.4/162.4 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.0/268.0 kB 26.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currentl

In [2]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [3]:
import os
import json
import subprocess
from typing import TypedDict, Annotated, List, Dict, Any
import operator

from langgraph.graph import StateGraph, END
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI

# 1. LLM Մոդելի սկզբնավորում
llm = ChatOpenAI(model="gpt-4o", temperature=0.1)


# 2. State-ի սահմանում
class AgentState(TypedDict):
    dialogue: Annotated[List[Dict[str, str]], operator.add]
    current_role: str
    instruction: str
    reason: str
    step_count: int

In [4]:
# 3. Executor Tool & Actuator
def execute_python_code(code: str, filename: str = "pinn_silverbox.py") -> str:
    with open(filename, "w", encoding="utf-8") as f:
        f.write(code)

    try:
        result = subprocess.run(
            ["python3", filename],
            capture_output=True,
            text=True,
            timeout=90
        )
        if result.returncode == 0:
            return f"SUCCESS (Exit code 0):\nSTDOUT:\n{result.stdout}"
        else:
            return f"ERROR (Exit code {result.returncode}):\nSTDOUT:\n{result.stdout}\nSTDERR:\n{result.stderr}"
    except subprocess.TimeoutExpired:
        return "ERROR: Execution timed out (exceeded 90 seconds)."
    except Exception as e:
        return f"EXECUTION FAILED: {str(e)}"

In [5]:
# 4. Agent Nodes-ի սահմանում

def orchestrator_node(state: AgentState):
    step_count = state.get("step_count", 0) + 1

    if step_count >= 8:
        return {
            "current_role": "finish",
            "instruction": "Maximum steps reached. Stop pipeline.",
            "reason": "Step limit reached.",
            "step_count": step_count
        }

    sys_prompt = """You are the Orchestrator for Task 5 Silverbox PINN on SNLS80mV.mat.
Analyze dialogue history and select next agent: 'scientist', 'general', 'executor', or 'finish'.

Output ONLY valid JSON:
- "next_role": "scientist" | "general" | "executor" | "finish"
- "instruction": "Clear directive"
- "reason": "Single sentence"

Rules:
1. Select 'executor' to write and run PyTorch code.
2. Select 'finish' ONLY if Executor prints "REAL TEST RMSE: <value>" successfully."""

    history_str = json.dumps(state["dialogue"], indent=2)
    response = llm.invoke([
        SystemMessage(content=sys_prompt),
        HumanMessage(content=f"Current Dialogue History:\n{history_str}")
    ])

    try:
        res_json = json.loads(response.content.strip().strip("```json").strip("```"))
        next_role = res_json.get("next_role", "finish")
        instruction = res_json.get("instruction", "")
        reason = res_json.get("reason", "")
    except Exception:
        next_role = "executor"
        instruction = "Write and execute PyTorch PINN code."
        reason = "Parsing fallback."

    return {
        "current_role": next_role,
        "instruction": instruction,
        "reason": reason,
        "step_count": step_count,
        "dialogue": [{"role": "Orchestrator", "content": f"Decision -> Role: {next_role} | Reason: {reason} | Instruction: {instruction}"}]
    }


def scientist_node(state: AgentState):
    sys_prompt = """You are the Scientist agent. Your job is to provide the exact mathematical plan and implementation details for Task 5:
- Circuit: Silverbox (Duffing oscillator), modeling output y and input u.
- File: 'SNLS80mV.mat' containing keys 'V1' (input u) and 'V2' (output y). No time 't' key exists.
- Data excerpt: Start at index 42650, take 4096 samples (3072 for training, 1024 for testing).
- Preprocessing: Center u and y by subtracting their means on this excerpt. Time step dt = 1.0 / 610.35.
- Model: A PyTorch multilayer fully connected network for acceleration v_dot = alpha(y, v, u).
- Integration: Classical Runge-Kutta 4th order (RK4) step with dt.
- Evaluation: Free-run test RMSE on 1024 test samples, printing 'REAL TEST RMSE: <value>' at the end."""

    instruction = state["instruction"]
    response = llm.invoke([
        SystemMessage(content=sys_prompt),
        HumanMessage(content=f"Instruction: {instruction}")
    ])
    return {"dialogue": [{"role": "Scientist", "content": response.content}]}


def general_node(state: AgentState):
    sys_prompt = "You are the General agent. Answer high-level questions."
    instruction = state["instruction"]
    response = llm.invoke([
        SystemMessage(content=sys_prompt),
        HumanMessage(content=f"Instruction: {instruction}")
    ])
    return {"dialogue": [{"role": "General Agent", "content": response.content}]}


def executor_node(state: AgentState):
    instruction = state["instruction"]
    history_str = json.dumps(state["dialogue"], indent=2)

    sys_prompt = """You are the Executor agent with a coding tool.
Write a complete, clean, and executable Python script using PyTorch to solve Task 5 (PINN for Silverbox) on 'SNLS80mV.mat'.
The script must build the PINN model with RK4 integration, train it, and print 'REAL TEST RMSE: <value>' at the end.
Output ONLY the executable Python code inside standard markdown code blocks (```python ... ```). Do not include conversational filler."""

    response = llm.invoke([
        SystemMessage(content=sys_prompt),
        HumanMessage(content=f"Instruction: {instruction}\n\nDialogue History:\n{history_str}")
    ])

    raw_content = response.content
    if "```python" in raw_content:
        generated_code = raw_content.split("```python")[1].split("```")[0].strip()
    elif "```" in raw_content:
        generated_code = raw_content.split("```")[1].split("```")[0].strip()
    else:
        generated_code = raw_content.strip()

    # Գործարկում ենք ակտուատորի միջոցով
    exec_result = execute_python_code(generated_code, filename="pinn_silverbox.py")

    return {
        "dialogue": [
            {"role": "Executor (Generated Code)", "content": generated_code},
            {"role": "Executor (Execution Result)", "content": exec_result}
        ]
    }

In [6]:
# 5. Routing Տրամաբանություն
def router(state: AgentState):
    role = state.get("current_role", "finish")
    if role == "scientist":
        return "scientist"
    elif role == "general":
        return "general"
    elif role == "executor":
        return "executor"
    else:
        return END


# 6. LangGraph Գրաֆի Կառուցում
builder = StateGraph(AgentState)

builder.add_node("orchestrator", orchestrator_node)
builder.add_node("scientist", scientist_node)
builder.add_node("general", general_node)
builder.add_node("executor", executor_node)

builder.set_entry_point("orchestrator")

builder.add_conditional_edges(
    "orchestrator",
    router,
    {"scientist": "scientist", "general": "general", "executor": "executor", END: END}
)

builder.add_edge("scientist", "orchestrator")
builder.add_edge("general", "orchestrator")
builder.add_edge("executor", "orchestrator")

app = builder.compile()
print("--- Agent System Setup Complete ---")

--- Agent System Setup Complete ---


In [7]:
initial_state = {
    "dialogue": [
        {
            "role": "User",
            "content": """Solve Task 5 (PINN for Silverbox circuit on SNLS80mV.mat).
Here are the key details:
- Use 'V1' for input u and 'V2' for output y.
- Slice excerpt from index 42650 to 46746 (4096 samples: first 3072 train, last 1024 test).
- Subtract mean from u and y.
- Compute dt = 1.0 / 610.35.
- Build PyTorch NN with RK4 integration.
- Train and evaluate free-run test RMSE, printing 'REAL TEST RMSE: <value>'."""
        }
    ],
    "current_role": "",
    "instruction": "",
    "reason": "",
    "step_count": 0
}

print("=== Starting LangGraph Agentic Pipeline ===")
for output in app.stream(initial_state):
    for node_name, state_update in output.items():
        print(f"\n--- Node: {node_name} ---")
        if "dialogue" in state_update:
            last_msg = state_update["dialogue"][-1]
            print(f"[{last_msg['role']}]: {last_msg['content'][:300]}...")

=== Starting LangGraph Agentic Pipeline ===

--- Node: orchestrator ---
[Orchestrator]: Decision -> Role: scientist | Reason: We need a clear plan for data preprocessing and model design before implementation. | Instruction: Outline the steps to preprocess the data and design the neural network architecture for the Silverbox circuit task....

--- Node: scientist ---
[Scientist]: To preprocess the data and design the neural network architecture for the Silverbox circuit task, follow these steps:

### Step 1: Load and Preprocess the Data
1. **Load the Data:**
   - Load the 'SNLS80mV.mat' file using a library like `scipy.io` to access the keys 'V1' (input u) and 'V2' (output y...

--- Node: orchestrator ---
[Orchestrator]: Decision -> Role: executor | Reason:  | Instruction: Implement the data preprocessing steps and design the neural network architecture as outlined by the scientist. Begin by loading the data, extracting the specified excerpt, centering the data, and defining the time st

In [8]:
with open('pinn_silverbox.py', 'r', encoding='utf-8') as f:
    print(f.read())

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.io import loadmat
from torch.utils.data import DataLoader, TensorDataset

# Load and preprocess the data
data = loadmat('SNLS80mV.mat')
u = data['V1'].flatten()
y = data['V2'].flatten()

# Extract the specified excerpt
start_idx = 42650
end_idx = 46746
u_excerpt = u[start_idx:end_idx]
y_excerpt = y[start_idx:end_idx]

# Split the data into training and testing
train_size = 3072
test_size = 1024

u_train = u_excerpt[:train_size]
y_train = y_excerpt[:train_size]
u_test = u_excerpt[train_size:]
y_test = y_excerpt[train_size:]

# Center the data
u_mean = np.mean(u_train)
y_mean = np.mean(y_train)

u_train_centered = u_train - u_mean
y_train_centered = y_train - y_mean
u_test_centered = u_test - u_mean
y_test_centered = y_test - y_mean

# Define the time step
dt = 1.0 / 610.35

# Define the neural network architecture
class DuffingNet(nn.Module):
    def __init__(self):
        super(DuffingNet, sel